In [1]:
import torch
import time

# 读取文本
input_file = "datasets/to-live-a-novel-cleaned.txt"
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()
print(f"text length: {len(text)}")

# 构建词汇表，建立字符与索引之间的双向映射
chars_list = sorted(list(set(text)))
char2idx = {c: i for i, c in enumerate(chars_list)}
idx2char = {i: c for i, c in enumerate(chars_list)}
vocab_size = len(chars_list)
print(f"词汇表大小: {vocab_size}")

# 解码器需要 <BOS> 起始符，其索引为 vocab_size
BOS_IDX = vocab_size
# 解码器嵌入矩阵大小需要 +1 以包含 <BOS>
decoder_vocab_size = vocab_size + 1


class CharDataset(torch.utils.data.Dataset):
    """数据集类，返回整数索引张量，嵌入操作推迟到模型内部执行"""

    def __init__(self, text, char2idx, learn_char_len=128, step_char_len=1):
        self.char2idx = char2idx
        self.vocab_size = len(char2idx)
        self.learn_char_len = learn_char_len
        # 将整篇文本转为索引列表
        self.data = [char2idx[c] for c in text if c in char2idx]
        self.samples = []
        # 用滑动窗口切出训练样本
        for i in range(0, len(self.data) - learn_char_len, step_char_len):
            x_idx = self.data[i : i + learn_char_len]
            y_idx = self.data[i + 1 : i + learn_char_len + 1]
            self.samples.append((x_idx, y_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_idx, y_idx = self.samples[idx]
        # 返回整数索引
        x_tensor = torch.tensor(x_idx, dtype=torch.long)
        y_tensor = torch.tensor(y_idx, dtype=torch.long)
        return x_tensor, y_tensor


class Seq2Seq(torch.nn.Module):
    """基于 Encoder-Decoder 架构的字符级序列模型"""

    def __init__(self, vocab_size, hidden_size, num_layers=1, embed_size=None):
        super().__init__()
        if embed_size is None:
            embed_size = hidden_size   # 嵌入维度默认与隐藏层相同

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 编码器部分
        self.encoder_embed = torch.nn.Embedding(vocab_size, embed_size)
        self.encoder_rnn = torch.nn.RNN(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh'
        )

        # 解码器部分（嵌入矩阵包含 <BOS>）
        self.decoder_embed = torch.nn.Embedding(vocab_size + 1, embed_size)
        self.decoder_rnn = torch.nn.RNN(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh'
        )

        # 输出层：将解码器隐状态映射回词汇表（不含 <BOS>）
        self.fc_out = torch.nn.Linear(hidden_size, vocab_size)

    def forward(self, encoder_input, decoder_input):
        """
        encoder_input: (batch, src_len)   编码器输入字符索引
        decoder_input: (batch, tgt_len)   解码器输入（开头包含 BOS_IDX）
        返回 logits: (batch, tgt_len, vocab_size)
        """
        # 编码器
        enc_emb = self.encoder_embed(encoder_input)          # (batch, src_len, embed)
        _, h_n = self.encoder_rnn(enc_emb)                   # h_n: (num_layers, batch, hidden)

        # 解码器
        dec_emb = self.decoder_embed(decoder_input)          # (batch, tgt_len, embed)
        dec_out, _ = self.decoder_rnn(dec_emb, h_n)          # (batch, tgt_len, hidden)
        logits = self.fc_out(dec_out)                        # (batch, tgt_len, vocab_size)
        return logits

    def generate(self, seed_indices, gen_len, device):
        """给定种子索引列表，生成指定长度的续写（贪心解码）"""
        self.eval()
        # 编码器：处理整个种子序列
        enc_input = torch.tensor(seed_indices, dtype=torch.long, device=device).unsqueeze(0)  # (1, src_len)
        enc_emb = self.encoder_embed(enc_input)
        _, h_n = self.encoder_rnn(enc_emb)   # h_n: (num_layers, 1, hidden)

        # 解码器初始输入为 <BOS>
        current_idx = torch.tensor([[BOS_IDX]], dtype=torch.long, device=device)  # (1, 1)
        h = h_n
        generated_indices = []

        for _ in range(gen_len):
            dec_emb = self.decoder_embed(current_idx)         # (1, 1, embed)
            out, h = self.decoder_rnn(dec_emb, h)             # out: (1, 1, hidden)
            logits = self.fc_out(out)                         # (1, 1, vocab_size)
            next_idx = logits.argmax(dim=-1).item()           # 贪心取最大概率字符
            generated_indices.append(next_idx)
            current_idx = torch.tensor([[next_idx]], dtype=torch.long, device=device)

        return generated_indices


# 选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# 初始化模型
hidden_size = 128
model = Seq2Seq(vocab_size=vocab_size, hidden_size=hidden_size).to(device)

# 构建数据集和数据加载器
char_dataset = CharDataset(text, char2idx=char2idx, learn_char_len=128)
dataloader = torch.utils.data.DataLoader(
    char_dataset, batch_size=64, num_workers=8, shuffle=True, pin_memory=True
)

# 初始化优化器和损失函数
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

total_batches = len(dataloader)
num_epochs = 3

# 训练循环
for epoch in range(num_epochs):
    total_loss = 0
    batch_count = 0
    for batch_idx, (x_idx_batch, y_idx_batch) in enumerate(dataloader):
        # 数据搬运到设备
        x_idx_batch = x_idx_batch.to(device, non_blocking=True)    # 编码器输入
        y_idx_batch = y_idx_batch.to(device, non_blocking=True)    # 目标序列

        batch_size = x_idx_batch.size(0)
        seq_len = x_idx_batch.size(1)

        # 构造解码器输入：开头为 <BOS>，后面是目标序列去掉最后一位
        bos_column = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=device)
        decoder_input = torch.cat([bos_column, y_idx_batch[:, :-1]], dim=1)   # (batch, seq_len)

        # 前向传播
        logits = model(x_idx_batch, decoder_input)   # (batch, seq_len, vocab_size)

        # 计算损失
        preds = logits.reshape(-1, vocab_size)
        targets = y_idx_batch.reshape(-1)
        loss = loss_fn(preds, targets)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        remaining_batches = total_batches - (batch_idx + 1)
        eta = remaining_batches * 0.09
        print(f"\rEpoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{total_batches}] | 损失: {loss.item():.4f} | 预计剩余: {eta:.1f}s", end="")

    print(f"\rEpoch [{epoch+1}/{num_epochs}] 完成 | 平均损失: {total_loss / batch_count:.4f}")


def generate_text(model, char2idx, idx2char, seed_text, gen_len=80):
    """给定种子文本，用 Encoder-Decoder 模型逐字符生成续写"""
    model.eval()
    device = next(model.parameters()).device

    # 将种子文本转换为索引
    seed_indices = [char2idx[c] for c in seed_text if c in char2idx]
    if not seed_indices:
        return seed_text  # 没有有效字符，直接返回

    # 用模型生成索引序列
    gen_indices = model.generate(seed_indices, gen_len, device)

    # 将索引转换回字符
    generated_chars = [idx2char[idx] for idx in gen_indices]
    return seed_text + "".join(generated_chars)


# 测试生成
seed = "凤霞命苦啊，"
text_gen = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text_gen}'")
print("-" * 50)

text length: 98634
词汇表大小: 1863
Using device: cuda
Epoch [1/3] 完成 | 平均损失: 2.98440] | 损失: 2.1868 | 预计剩余: 0.0ss
Epoch [2/3] 完成 | 平均损失: 1.90750] | 损失: 1.7198 | 预计剩余: 0.0ss
Epoch [3/3] 完成 | 平均损失: 1.54050] | 损失: 1.4062 | 预计剩余: 0.0ss

种子: '凤霞命苦啊，'
生成: '凤霞命苦啊，，我心里一阵酸疼，我想这可是你爹年轻时，便打了一阵后，我问："他娘的，每次都不知道该怎么办，那油可很多人的派头去向我背脊上了。<EOS>我们两个人都坐在晒场上，'
--------------------------------------------------


In [3]:
# 测试生成
seed = "你好"
text_gen = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text_gen}'")
print("-" * 50)


种子: '你好'
生成: '你好着说她，城里的活声音都青了一惊，悄悄抽了抽，心想我就得不行了，我们陈家安女人，我们把我从前宰了，我和二喜一会，我就问我："你要是县太爷的公子，就向我娘的坟上没了'
--------------------------------------------------
